In [5]:
from sklearn.model_selection import StratifiedShuffleSplit
import time
import pandas as pd
import numpy as np
from sklearn.metrics import (accuracy_score, roc_auc_score, recall_score, precision_score, f1_score, 
                           cohen_kappa_score, matthews_corrcoef)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifier, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [6]:

# 1. Load and split the original dataset
whole_Set = pd.read_excel("D:\\AncestryGeni\\PredictRace\\new\\rf_training_set.xlsx")

# Create stratified 80-20 split
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

# Separate features and target
X = whole_Set.drop(['SampleCode', 'Code'], axis=1)
y = whole_Set['Code']

for train_index, test_index in sss.split(X, y):
    # Save the fixed training set
    train_data = whole_Set.iloc[train_index]
    train_data.to_excel("D:\\AncestryGeni\\PredictRace\\new\\fixed_train_set.xlsx", index=False)
    print("Saved fixed training set")
    
    # Get test sample IDs
    test_sample_codes = whole_Set.iloc[test_index]['SampleCode']
    
    # Get test data from each SNP density file
    for file in ["300K.txt", "50K.txt", "10K.txt", "1K.txt", "100.txt"]:
        # Read the SNP file
        current_data = pd.read_csv(f"D:\\AncestryGeni\\Data\\Input_1KG\\{file}", delimiter='\t')
        
        # Extract only the test samples using SampleCodes
        test_data = current_data[current_data['SampleCode'].isin(test_sample_codes)]
        
        # Save test set
        test_output = f"D:\\AncestryGeni\\PredictRace\\new\\test_set_{file.split('.')[0]}.xlsx"
        test_data.to_excel(test_output, index=False)
        print(f"Saved: {test_output}")

Saved fixed training set
Saved: D:\AncestryGeni\PredictRace\new\test_set_300K.xlsx
Saved: D:\AncestryGeni\PredictRace\new\test_set_50K.xlsx
Saved: D:\AncestryGeni\PredictRace\new\test_set_10K.xlsx
Saved: D:\AncestryGeni\PredictRace\new\test_set_1K.xlsx
Saved: D:\AncestryGeni\PredictRace\new\test_set_100.xlsx


In [7]:
# Define models
models = {
    'SVM - Linear Kernel': SVC(kernel='linear', probability=True),
    'Ridge Classifier': RidgeClassifier(),
    'Random Forest Classifier': RandomForestClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Linear Discriminant Analysis': LinearDiscriminantAnalysis(),
    'Gradient Boosting Classifier': GradientBoostingClassifier(random_state=42),
    'Light Gradient Boosting Machine': LGBMClassifier(random_state=42),
    'Naive Bayes': GaussianNB(),
    'Extreme Gradient Boosting': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'Extra Trees Classifier': ExtraTreesClassifier(random_state=42),
    'K Neighbors Classifier': KNeighborsClassifier(),
    'Decision Tree Classifier': DecisionTreeClassifier(random_state=42),
    'Quadratic Discriminant Analysis': QuadraticDiscriminantAnalysis(),
    'Ada Boost Classifier': AdaBoostClassifier(random_state=42)
}

# Load training data
train_data = pd.read_excel("D:\\AncestryGeni\\PredictRace\\new\\fixed_train_set.xlsx")

# List of test files
test_files = ["300K", "50K", "10K", "1K", "100"]

# Process each test file
for test_name in test_files:
    print(f"\nProcessing test set: {test_name}")
    
    # Load test data
    test_data = pd.read_excel(f"D:\\AncestryGeni\\PredictRace\\new\\test_set_{test_name}.xlsx")
    
    # For training data - drop SampleCode and Code
    X_train = train_data.drop(columns=["SampleCode", "Code"])
    y_train = train_data["Code"]

    # For test data - drop Dataset, Sample, SampleCode and Code
    X_test = test_data.drop(columns=["Dataset", "Sample", "SampleCode", "Code"])
    y_test = test_data["Code"]

    # Encode target variables
    label_encoder = LabelEncoder()
    y_train = label_encoder.fit_transform(y_train)
    y_test = label_encoder.transform(y_test)

    # Standardize the data
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # Store results
    results = []

    # Evaluate each model
    for name, model in models.items():
        print(f"Training {name}...")
        start_time = time.time()
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate AUC
        try:
            if hasattr(model, "predict_proba"):
                y_prob = model.predict_proba(X_test)
                auc = roc_auc_score(y_test, y_prob, multi_class="ovr")
            else:
                auc = np.nan
        except Exception:
            auc = np.nan

        # Calculate metrics
        accuracy = accuracy_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred, average='weighted')
        precision = precision_score(y_test, y_pred, average='weighted')
        f1 = f1_score(y_test, y_pred, average='weighted')
        kappa = cohen_kappa_score(y_test, y_pred)
        mcc = matthews_corrcoef(y_test, y_pred)
        train_time = time.time() - start_time

        # Store results
        results.append({
            'Model': name,
            'Accuracy': accuracy,
            'AUC': auc,
            'Recall': recall,
            'Precision': precision,
            'F1': f1,
            'Kappa': kappa,
            'MCC': mcc,
            'TT (Sec)': round(train_time, 4)
        })

    # Create and save results DataFrame
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(by='Accuracy', ascending=False).reset_index(drop=True)
    results_df.to_excel(f"D:\\AncestryGeni\\PredictRace\\new\\model_comparison_results_{test_name}.xlsx", index=False)
    print(f"Saved results for {test_name}")

print("\nCompleted all comparisons!")


Processing test set: 300K
Training SVM - Linear Kernel...
Training Ridge Classifier...
Training Random Forest Classifier...
Training Logistic Regression...
Training Linear Discriminant Analysis...
Training Gradient Boosting Classifier...
Training Light Gradient Boosting Machine...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000256 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 504
[LightGBM] [Info] Number of data points in the train set: 540, number of used features: 12
[LightGBM] [Info] Start training from score -1.676449
[LightGBM] [Info] Start training from score -1.222665
[LightGBM] [Info] Start training from score -1.909543
[LightGBM] [Info] Start training from score -0.993252
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best g

c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:24:23] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Extra Trees Classifier...
Training K Neighbors Classifier...
Training Decision Tree Classifier...
Training Quadratic Discriminant Analysis...
Training Ada Boost Classifier...


c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Saved results for 300K

Processing test set: 50K
Training SVM - Linear Kernel...
Training Ridge Classifier...
Training Random Forest Classifier...
Training Logistic Regression...
Training Linear Discriminant Analysis...
Training Gradient Boosting Classifier...
Training Light Gradient Boosting Machine...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000334 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 504
[LightGBM] [Info] Number of data points in the train set: 540, number of used features: 12
[LightGBM] [Info] Start training from score -1.676449
[LightGBM] [Info] Start training from score -1.222665
[LightGBM] [Info] Start training from score -1.909543
[LightGBM] [Info] Start training from score -0.993252
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with

...

c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


Training Logistic Regression...
Training Linear Discriminant Analysis...
Training Gradient Boosting Classifier...
Training Light Gradient Boosting Machine...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000451 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 504
[LightGBM] [Info] Number of data points in the train set: 540, number of used features: 12
[LightGBM] [Info] Start training from score -1.676449
[LightGBM] [Info] Start training from score -1.222665
[LightGBM] [Info] Start training from score -1.909543
[LightGBM] [Info] Start training from score -0.993252
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits w

c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [23:24:27] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-0ed59c031377d09b8-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


Training Extra Trees Classifier...
Training K Neighbors Classifier...
Training Decision Tree Classifier...
Training Quadratic Discriminant Analysis...
Training Ada Boost Classifier...
Saved results for 100

Completed all comparisons!


c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\discriminant_analysis.py:947: UserWarning: Variables are collinear
  warnings.warn("Variables are collinear")
c:\Users\sarabh\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
